# Multimodal Notes RAG

This notebook ingests PDFs and supported images from `data/raw/`, then retrieves note passages and generates a source-grounded answer. Run it from a project environment created with `uv sync`, and configure `PINECONE_API_KEY`, `PINECONE_INDEX_NAME`, and `GROQ_API_KEY` in the project-root `.env` file first.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find the project root (pyproject.toml) from the current directory.")

src_path = str(project_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
load_dotenv(project_root / ".env")

required_settings = ("PINECONE_API_KEY", "PINECONE_INDEX_NAME", "GROQ_API_KEY")
missing_settings = [name for name in required_settings if not os.getenv(name)]
if missing_settings:
    raise RuntimeError(f"Set these values in the project-root .env file: {', '.join(missing_settings)}")

pdf_dir = project_root / "data" / "raw" / "pdfs"
image_dir = project_root / "data" / "raw" / "images"
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".webp"}
pdf_files = sorted(pdf_dir.glob("*.pdf"))
image_files = sorted(
    path for path in image_dir.iterdir()
    if path.is_file() and path.suffix.lower() in image_extensions
) if image_dir.is_dir() else []

print(f"Project root: {project_root}")
print(f"PDF files: {len(pdf_files)}; image files: {len(image_files)}")

## Ingest notes

Add source PDFs or images to the folders shown above. This step extracts text (using OCR for image content), chunks and embeds it, and writes the vectors and source metadata to the configured Pinecone index. Re-run it whenever you add or change files.

In [ ]:
from HandwrittenNotesRAG.pipelines.ingestion_pipeline import IngestionPipeline

if not pdf_files and not image_files:
    ingestion_result = {"files": 0, "documents": 0, "chunks": 0, "stored": 0}
    print("No PDFs or supported images found. Add files to data/raw/pdfs or data/raw/images, then rerun this cell.")
else:
    ingestion_result = IngestionPipeline().ingest(pdf_dir=pdf_dir, image_dir=image_dir)
    print(ingestion_result)
    if ingestion_result["stored"] == 0:
        print("No text chunks were stored; check that the source files contain extractable text.")

## Ask a question

Edit `question` below. The answer uses retrieved passages and returns those passages with their source filenames and page numbers when available.

In [ ]:
from HandwrittenNotesRAG.pipelines.rag_pipeline import RAGPipeline

question = "What is a recurrence relation?"
result = RAGPipeline().ask(question)

print("Answer:\n")
print(result["answer"])

print("\nRetrieved sources:")
for number, source in enumerate(result["sources"], start=1):
    label = source.get("source") or "Unknown source"
    page = source.get("page")
    if page is not None:
        label += f", page {page}"
    print(f"\n[{number}] {label}")
    print(source.get("text", ""))